# Production Evaluation — Config D (Recursive + Hybrid + Rerank)

This notebook re-evaluates Config D on the **production system** — meaning:
- Filtered chunks (TOC and amendment noise removed)
- Query rewriting enabled (bridges vocabulary gap)

This produces a second data point for the ablation table showing
the improvement from production optimisations over the baseline.

**Upload before running:**
- `chunks_fixed.json` (filtered version)
- `chunks_recursive.json` (filtered version)

**Runtime:** T4 GPU

## Cell 1 — Install

In [1]:
!pip install -q pinecone sentence-transformers rank-bm25 groq tqdm numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 31.2 MB/s eta 0:00:00


## Cell 2 — Config (EDIT THIS)

In [ ]:
PINECONE_API_KEY = "KEY"
PINECONE_INDEX   = "pakistani-law"
GROQ_API_KEY     = "KEY"

MODEL            = "llama-3.3-70b-versatile"
EMBEDDING_MODEL  = "all-MiniLM-L6-v2"
CROSSENCODER     = "cross-encoder/ms-marco-MiniLM-L-6-v2"

NS_RECURSIVE     = "recursive"
TOP_K            = 5
SLEEP_BETWEEN    = 3   # seconds between LLM calls — avoids rate limiting

print("Config loaded.")

Config loaded.


## Cell 3 — Imports and Load Everything

In [3]:
import json, re, time, numpy as np
from tqdm import tqdm
from groq import Groq
from pinecone import Pinecone
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Load chunk files
with open("chunks_fixed.json", "r", encoding="utf-8") as f:
    fixed_chunks = json.load(f)
with open("chunks_recursive.json", "r", encoding="utf-8") as f:
    recursive_chunks = json.load(f)
print(f"Chunks: fixed={len(fixed_chunks)}, recursive={len(recursive_chunks)}")

# Load models
print("Loading bi-encoder...")
bi_encoder    = SentenceTransformer(EMBEDDING_MODEL, device=device)
print("Loading CrossEncoder...")
cross_encoder = CrossEncoder(CROSSENCODER, device=device)
print("Models loaded.")

# Build BM25 — recursive only (Config D)
print("Building BM25...")
bm25_recursive = BM25Okapi([c["text"].lower().split() for c in recursive_chunks])
print("BM25 ready.")

# Connect Pinecone
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)
stats = index.describe_index_stats()
print(f"Pinecone: {stats.total_vector_count} vectors")
print(f"Namespaces: {list(stats.namespaces.keys())}")

# Connect Groq
groq_client = Groq(api_key=GROQ_API_KEY)
print("All systems ready.")

Device: cuda
Chunks: fixed=4138, recursive=4768
Loading bi-encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Models loaded.
Building BM25...
BM25 ready.
Pinecone: 8906 vectors
Namespaces: ['fixed', 'recursive']
All systems ready.


## Cell 4 — LLM Call

In [4]:
def call_llm(prompt: str, max_tokens: int = 512,
             temperature: float = 0.1) -> str:
    for attempt in range(2):
        try:
            resp = groq_client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=max(temperature, 0.01),
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt == 0:
                print(f"  LLM retry: {str(e)[:80]}")
                time.sleep(5)
            else:
                print(f"  LLM failed: {str(e)[:80]}")
                return ""

# Sanity check
test = call_llm("Reply with OK only.", max_tokens=5)
print(f"LLM test: {test}")

LLM test: OK


## Cell 5 — Query Rewriting

In [5]:
def rewrite_query(query: str) -> str:
    """
    Rewrite query using Pakistani legal terminology.
    This is the production enhancement that was absent
    during the original ablation study.
    """
    prompt = f"""You are a Pakistani legal expert.
Rewrite this question using precise Pakistani legal terminology.
Include both English terms AND specific legal terms used in Pakistani law.
Return ONLY the rewritten query terms. No explanation. No prefix.
Maximum 20 words.

Examples:
Input: murder punishment
Output: qatl-i-amd punishment death imprisonment qisas diyat Section 302 PPC

Input: theft penalty
Output: theft dishonestly takes property Section 378 379 PPC

Input: terrorism definition
Output: terrorist act scheduled offence Anti-Terrorism Act 1997

Input: fundamental rights
Output: fundamental rights Article 8 9 10 25 Constitution Pakistan

Input: {query}
Output:"""

    rewritten = call_llm(prompt, max_tokens=40, temperature=0.0)
    rewritten = re.sub(
        r'^(rewritten|query|output|answer|here|terms?|result)[:\s]+',
        '', rewritten, flags=re.IGNORECASE
    ).strip()
    if not rewritten or len(rewritten) > 200:
        return query
    return rewritten


# Test
test_q = "What is the punishment for murder under the Pakistan Penal Code?"
print(f"Original:  {test_q}")
print(f"Rewritten: {rewrite_query(test_q)}")

Original:  What is the punishment for murder under the Pakistan Penal Code?
Rewritten: qatl-i-amd punishment death imprisonment qisas diyat Section 302 PPC


## Cell 6 — Retrieval Pipeline (Config D: Recursive + Hybrid + Rerank)

In [6]:
def retrieve_production(query: str, top_k: int = 5) -> dict:
    """
    Production retrieval: Config D + query rewriting.
    Recursive chunks, BM25 + Semantic + RRF + CrossEncoder.
    """
    t0 = time.time()

    # Step 1 — Rewrite query
    retrieval_query = rewrite_query(query)
    time.sleep(SLEEP_BETWEEN)

    # Step 2 — BM25
    bm25_scores  = bm25_recursive.get_scores(retrieval_query.lower().split())
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:20]
    bm25_results = []
    for rank, idx in enumerate(top_bm25_idx):
        r = dict(recursive_chunks[idx])
        r["bm25_rank"] = rank + 1
        bm25_results.append(r)

    # Step 3 — Semantic
    vec  = bi_encoder.encode(retrieval_query, convert_to_numpy=True).tolist()
    resp = index.query(vector=vec, top_k=20,
                       namespace=NS_RECURSIVE, include_metadata=True)
    sem_results = []
    for rank, m in enumerate(resp.matches):
        sem_results.append({
            "id":            m.id,
            "text":          m.metadata.get("text", ""),
            "source":        m.metadata.get("source", ""),
            "year":          m.metadata.get("year", ""),
            "semantic_rank": rank + 1,
        })

    # Step 4 — RRF
    all_chunks     = {r["id"]: r for r in bm25_results}
    all_chunks.update({r["id"]: r for r in sem_results})
    bm25_ranks     = {r["id"]: r["bm25_rank"]     for r in bm25_results}
    sem_ranks      = {r["id"]: r["semantic_rank"] for r in sem_results}
    all_ids        = set(bm25_ranks) | set(sem_ranks)
    K              = 60
    rrf_scores     = {
        did: (1/(bm25_ranks.get(did, 1e9)+K)) +
             (1/(sem_ranks.get(did, 1e9)+K))
        for did in all_ids
    }
    top_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:20]
    fused   = []
    for rank, did in enumerate(top_ids):
        c = dict(all_chunks[did])
        c["rrf_score"] = rrf_scores[did]
        c["rrf_rank"]  = rank + 1
        fused.append(c)

    # Step 5 — CrossEncoder rerank
    pairs  = [(retrieval_query, c["text"]) for c in fused]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    for i, c in enumerate(fused):
        c["crossencoder_score"] = float(scores[i])
    final = sorted(fused,
                   key=lambda x: x["crossencoder_score"],
                   reverse=True)[:top_k]

    context_parts = [
        f"[Source {i+1}: {c['source']} ({c['year']})]\n{c['text']}"
        for i, c in enumerate(final)
    ]
    context = "\n\n".join(context_parts)

    return {
        "chunks":           final,
        "context":          context,
        "retrieval_query":  retrieval_query,
        "latency_ms":       round((time.time() - t0) * 1000, 1)
    }

print("Retrieval function defined.")

Retrieval function defined.


## Cell 7 — Generation and Evaluation Functions

In [7]:
def generate_answer(query: str, context: str) -> str:
    prompt = f"""You are a legal assistant for Pakistani law.
You must follow these rules strictly:

RULE 1: Answer ONLY using information explicitly present in the context below.
RULE 2: If the context does not contain the answer, respond with exactly:
        "The provided legal documents do not contain sufficient information."
RULE 3: Do NOT use your own knowledge under any circumstances.
RULE 4: Do NOT say "based on general knowledge".
RULE 5: Do NOT narrate what the context lacks.

Context:
{context}

Question: {query}

Answer strictly from the context:"""
    return call_llm(prompt, max_tokens=600)


def extract_claims(answer: str) -> list:
    prompt = f"""You are a precise claim extractor.
Extract every distinct factual claim from the following text.
Return ONLY a numbered list, one claim per line.
Each claim must be a complete standalone sentence.

Text:
{answer}

Numbered list of factual claims:"""
    response = call_llm(prompt, max_tokens=400, temperature=0.0)
    claims = []
    for line in response.strip().split("\n"):
        match = re.match(r"^\d+[.)\s]+(.+)$", line.strip())
        if match and len(match.group(1).strip()) > 10:
            claims.append(match.group(1).strip())
    return claims


def verify_claim(claim: str, context: str) -> dict:
    prompt = f"""You are a fact-checker for legal documents.
Determine if the following claim is supported by the provided context.
Answer with ONLY 'YES' or 'NO' on the first line, then one sentence of reasoning.

Context:
{context[:2000]}

Claim: {claim}

Is this claim supported by the context?"""
    response   = call_llm(prompt, max_tokens=100, temperature=0.0)
    first_line = response.strip().split("\n")[0].strip().upper()
    supported  = "YES" in first_line and "NO" not in first_line[:3]
    return {"claim": claim, "supported": supported, "reasoning": response.strip()}


def compute_faithfulness(answer: str, context: str) -> dict:
    claims = extract_claims(answer)
    if not claims:
        return {"score": 0.0, "claims": [], "n_claims": 0, "n_supported": 0}
    time.sleep(SLEEP_BETWEEN)
    verifications = []
    for claim in claims:
        verifications.append(verify_claim(claim, context))
        time.sleep(SLEEP_BETWEEN)
    n_supported = sum(1 for v in verifications if v["supported"])
    return {
        "score":       round(n_supported / len(verifications), 4),
        "claims":      verifications,
        "n_claims":    len(verifications),
        "n_supported": n_supported
    }


def compute_relevancy(query: str, answer: str) -> dict:
    prompt = f"""You are given an answer to a legal question about Pakistani law.
Generate exactly 3 different questions that this answer could be responding to.
Return ONLY the 3 questions as a numbered list. Nothing else.

Answer:
{answer}

3 questions:"""
    response  = call_llm(prompt, max_tokens=200, temperature=0.3)
    questions = []
    for line in response.strip().split("\n"):
        match = re.match(r"^\d+[.)\s]+(.+\?)$", line.strip())
        if match and len(match.group(1)) > 10:
            questions.append(match.group(1).strip())
        elif line.strip().endswith("?") and len(line.strip()) > 10:
            questions.append(line.strip())
    questions = questions[:3]
    if not questions:
        return {"score": 0.0, "generated_questions": [], "similarities": []}

    query_vec = bi_encoder.encode(query,     convert_to_numpy=True)
    gen_vecs  = bi_encoder.encode(questions, convert_to_numpy=True)
    sims = []
    for gv in gen_vecs:
        q_n = query_vec / (np.linalg.norm(query_vec) + 1e-10)
        g_n = gv        / (np.linalg.norm(gv)        + 1e-10)
        sims.append(round(float(np.dot(q_n, g_n)), 4))
    return {
        "score":               round(float(np.mean(sims)), 4),
        "generated_questions": questions,
        "similarities":        sims
    }


print("All evaluation functions defined.")

All evaluation functions defined.


## Cell 8 — Fixed Test Set (same 15 queries as ablation study)

In [8]:
TEST_SET = [
    "What is the punishment for murder under the Pakistan Penal Code?",
    "How does the PPC define theft and what is its penalty?",
    "What constitutes defamation under Pakistani criminal law?",
    "What fundamental rights does the Constitution of Pakistan guarantee?",
    "What are the qualifications required to become President of Pakistan?",
    "How does Pakistani law define a terrorist act?",
    "What are the special powers of courts under the Anti-Terrorism Act?",
    "What are the essential elements of a valid contract in Pakistan?",
    "When is a contract considered void under the Contract Act 1872?",
    "What offences fall under the National Accountability Ordinance?",
    "What is the procedure for filing a First Information Report in Pakistan?",
    "What powers does a magistrate have under the Code of Criminal Procedure?",
    "What types of evidence are admissible in Pakistani courts?",
    "What are the legal requirements for divorce under Muslim Family Laws?",
    "What are the duties of directors under the Companies Act 2017?",
]

print(f"Test set: {len(TEST_SET)} queries")
print("These are identical to the original ablation study queries.")

Test set: 15 queries
These are identical to the original ablation study queries.


## Cell 9 — Run Production Evaluation

This runs Config D on the production system.
Estimated time: 30-50 minutes.
Results saved after every query — no data loss if session dies.

In [9]:
results      = []
partial_save = "production_D_partial.json"

print("=" * 60)
print("PRODUCTION EVAL: Config D + Query Rewriting + Filtered Chunks")
print("=" * 60)

for i, query in enumerate(tqdm(TEST_SET, desc="Evaluating")):
    try:
        # 1 — Retrieve (includes query rewriting internally)
        retrieval   = retrieve_production(query, top_k=TOP_K)
        rewritten_q = retrieval["retrieval_query"]
        time.sleep(SLEEP_BETWEEN)

        # 2 — Generate
        t0     = time.time()
        answer = generate_answer(query, retrieval["context"])
        gen_ms = round((time.time() - t0) * 1000, 1)
        time.sleep(SLEEP_BETWEEN)

        # 3 — Faithfulness
        faith = compute_faithfulness(answer, retrieval["context"])
        time.sleep(SLEEP_BETWEEN)

        # 4 — Relevancy
        relev = compute_relevancy(query, answer)
        time.sleep(SLEEP_BETWEEN)

        total_ms = retrieval["latency_ms"] + gen_ms

        result = {
            "query":              query,
            "retrieval_query":    rewritten_q,
            "answer":             answer,
            "context":            retrieval["context"],
            "faithfulness_score": faith["score"],
            "faithfulness_detail":faith,
            "relevancy_score":    relev["score"],
            "relevancy_detail":   relev,
            "retrieval_ms":       retrieval["latency_ms"],
            "generation_ms":      gen_ms,
            "total_ms":           total_ms,
            "config":             "D_production_with_rewriting"
        }
        results.append(result)

        print(f"  Q{i+1:02d}: Faith={faith['score']:.2%}  "
              f"Relev={relev['score']:.4f}  "
              f"Time={total_ms:.0f}ms")
        print(f"        Rewritten: {rewritten_q[:80]}")

        # Save after every query
        with open(partial_save, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    except Exception as e:
        print(f"  Q{i+1:02d}: ERROR — {e}")
        results.append({"query": query, "error": str(e)})

print("\n=== EVALUATION COMPLETE ===")

PRODUCTION EVAL: Config D + Query Rewriting + Filtered Chunks


Evaluating:   7%|▋         | 1/15 [00:31<07:23, 31.71s/it]

  Q01: Faith=100.00%  Relev=0.7663  Time=5295ms
        Rewritten: qatl-i-amd punishment death imprisonment qisas diyat Section 302 PPC


Evaluating:  13%|█▎        | 2/15 [00:58<06:11, 28.59s/it]

  Q02: Faith=0.00%  Relev=0.6084  Time=3933ms
        Rewritten: Theft definition Section 378 PPC, punishment Section 379 PPC, imprisonment, fine


...

  Q13: Faith=42.86%  Relev=0.7587  Time=4235ms
        Rewritten: Primary secondary evidence admissible daleel Section 60 61 Qanoon-e-Shahadat Ord


Evaluating:  93%|█████████▎| 14/15 [08:11<00:32, 32.11s/it]

  Q14: Faith=0.00%  Relev=0.3510  Time=3900ms
        Rewritten: dissolution marriage talaq khula faskh Section 7 8 Muslim Family Laws Ordinance 


Evaluating: 100%|██████████| 15/15 [08:55<00:00, 35.70s/it]

  Q15: Faith=71.43%  Relev=0.7606  Time=4248ms
        Rewritten: Directors' duties fiduciary duties Section 157 158 Companies Act 2017.

=== EVALUATION COMPLETE ===


## Cell 10 — Summary and Comparison

In [11]:
valid = [r for r in results if "faithfulness_score" in r]

avg_faith = round(np.mean([r["faithfulness_score"] for r in valid]), 4)
avg_relev = round(np.mean([r["relevancy_score"]    for r in valid]), 4)
avg_total = round(np.mean([r["total_ms"]           for r in valid]), 1)
avg_retr  = round(np.mean([r["retrieval_ms"]       for r in valid]), 1)
avg_gen   = round(np.mean([r["generation_ms"]      for r in valid]), 1)

faith_str = f"{avg_faith:.2%}"
relev_str = f"{avg_relev:.4f}"
total_str = f"{avg_total:,.0f}"

print("=" * 65)
print("PRODUCTION RESULTS vs ABLATION BASELINE")
print("=" * 65)
print(f"{'Metric':<30} {'Baseline (D)':>15} {'Production (D)':>15}")
print("-" * 65)
print(f"{'Faithfulness':<30} {'48.31%':>15} {faith_str:>15}")
print(f"{'Relevancy':<30} {'0.7831':>15} {relev_str:>15}")
print(f"{'Avg Total Latency (ms)':<30} {'12,663':>15} {total_str:>15}")
print("-" * 65)
print(f"\nBreakdown (production):")
print(f"  Avg retrieval time:  {avg_retr:.0f}ms")
print(f"  Avg generation time: {avg_gen:.0f}ms")
print(f"  Valid queries: {len(valid)}/{len(TEST_SET)}")

improvement_faith = round((avg_faith - 0.4831) * 100, 2)
print(f"\nFaithfulness improvement: {improvement_faith:+.2f} percentage points")

PRODUCTION RESULTS vs ABLATION BASELINE
Metric                            Baseline (D)  Production (D)
-----------------------------------------------------------------
Faithfulness                            48.31%          51.87%
Relevancy                               0.7831          0.6277
Avg Total Latency (ms)                  12,663           4,087
-----------------------------------------------------------------

Breakdown (production):
  Avg retrieval time:  3607ms
  Avg generation time: 480ms
  Valid queries: 15/15

Faithfulness improvement: +3.56 percentage points


## Cell 11 — Save Final Results and Download

In [12]:
final_output = {
    "config":          "D_production_with_rewriting",
    "description":     "Config D (Recursive + Hybrid + Rerank) with production "
                       "optimisations: TOC-filtered chunks + LLM query rewriting",
    "n_queries":       len(TEST_SET),
    "n_valid":         len(valid),
    "avg_faithfulness":avg_faith,
    "avg_relevancy":   avg_relev,
    "avg_retrieval_ms":avg_retr,
    "avg_generation_ms":avg_gen,
    "avg_total_ms":    avg_total,
    "baseline_comparison": {
        "faithfulness_baseline":   0.4831,
        "faithfulness_production": avg_faith,
        "relevancy_baseline":      0.7831,
        "relevancy_production":    avg_relev,
    },
    "per_query": results
}

with open("production_D_final.json", "w", encoding="utf-8") as f:
    json.dump(final_output, f, ensure_ascii=False, indent=2)
print("Saved production_D_final.json")

from google.colab import files
files.download("production_D_final.json")
print("Downloaded.")

Saved production_D_final.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded.
